In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

CODE_PRECURATED = os.environ["GPTNL_CODE_PRECURATED"]
CODE_POSTCURATED = os.environ["GPTNL_CODE_POSTCURATED"]

In [6]:
import os
import random


# Function to sample parquet files from a directory
def sample_parquet_files(directory, sample_size=5):
    parquet_files = []
    for root, _, files in os.walk(directory):
        for file in files:
            if file.endswith(".parquet"):
                parquet_files.append(os.path.join(root, file))

    if not parquet_files:
        return []

    return random.sample(parquet_files, min(sample_size, len(parquet_files)))


# Sample files from both directories
precurated_files = sample_parquet_files(CODE_PRECURATED, sample_size=100)
postcurated_files = sample_parquet_files(CODE_POSTCURATED, sample_size=100)

In [7]:
import numpy as np
import pyarrow.parquet as pq


# Function to read parquet files and count characters
def count_chars_in_parquet(file_path):
    try:
        # Read only the 'text' column to save memory
        table = pq.read_table(file_path, columns=["text"])
        df = table.to_pandas()

        # Count characters in each row
        char_counts = df["text"].str.len()

        return char_counts.tolist()
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return []


# Process all precurated files
precurated_chars = []
for file in precurated_files:
    chars = count_chars_in_parquet(file)
    precurated_chars.extend(chars)

# Process all postcurated files
postcurated_chars = []
for file in postcurated_files:
    chars = count_chars_in_parquet(file)
    postcurated_chars.extend(chars)

# Calculate statistics
precurated_mean = np.mean(precurated_chars)
postcurated_mean = np.mean(postcurated_chars)
postcurated_std = np.std(postcurated_chars)
precurated_std = np.std(precurated_chars)

# Print results
print(
    f"PRECURATED - Mean number of chars: {precurated_mean:.2f} (Std: {precurated_std:.2f})"
)
print(
    f"POSTCURATED - Mean number of chars: {postcurated_mean:.2f} (Std: {postcurated_std:.2f})"
)

PRECURATED - Mean number of chars: 4096.41 (Std: 13589.96)
POSTCURATED - Mean number of chars: 2387.57 (Std: 5073.06)
